# PointNet++ Training experimental

This notebook serves the purpose of creating the training routine for the PointNet++ to encode point clouds into the same latent space as the CAD sequence encoding done by the DeepCAD model.

The pointnet venv is located at "Point-Cloud-Reconstruction/pointnet.pytorch", but the model used is in "Point-Cloud-Reconstruction/Pointnet_Pointnet2_pytorch"

## 1. Check point cloud data
We will use open3d to inspect the point cloud files. During the conversion of the CAD sequences to point clouds errors appeared, which is why I assume there must be some corrupt point cloud files, which need to be excluded.

- Load train/val/test split
- create train/val/test pc path lists

In [38]:
import open3d as o3d
import os
from tqdm import tqdm
from glob import glob

In [39]:
DATA_ROOT = "../data"

PC_ROOT = os.path.join(DATA_ROOT, "pc_cad")
SPLIT = "../data/train_val_test_split.json"

Here the split file is opened, it contains the information about all files in the dataset

In [40]:
with open(SPLIT, "r") as fp:
    all_data = json.load(fp)
print(f"Number of samples that should be in the split: {len(all_data['train']) + len(all_data['validation']) + len(all_data['test'])}")
for phase in all_data.keys():
    print(phase, len(all_data[phase]))

Number of samples that should be in the split: 178238
train 161240
validation 8946
test 8052


In [57]:
train = [os.path.join(PC_ROOT, f"{idx}.ply") for idx in all_data['train']]
val = [os.path.join(PC_ROOT, f"{idx}.ply") for idx in all_data['validation']]
test = [os.path.join(PC_ROOT, f"{idx}.ply") for idx in all_data['test']]

Now we need to check if the point cloud files in the split are actually there and not corrupt. The point cloud files are generated using json2pc.py, which can fail from time to time.

In [58]:
file_pattern = "**/*.ply"
all_files = glob(f"{PC_ROOT}/{file_pattern}", recursive=True)
print(f"Total number of files: {len(all_files)}")
print(f"Missing files: {len(train)+len(val)+len(test)-len(all_files)}")

Total number of files: 177948
Missing files: 290


Apparently in 290 cases no point cloud file was created. These have to be removed from the split.

In [43]:
all_files_set = set(all_files)
train = [entry for entry in train if entry in all_files_set]
val = [entry for entry in val if entry in all_files_set]
test = [entry for entry in test if entry in all_files_set]

In [47]:
assert (len(train)+len(val)+len(test) == len(all_files))

In [54]:
def check_valid_pc(paths_list):
    corrupt_counter = 0
    for pc_file in tqdm(paths_list):
        try:
            pc = o3d.io.read_point_cloud(pc_file)
            if not pc.has_points():
                corrupt_counter += 1
        except Exception as e:
            corrupt_counter += 1
    print(f"There are {corrupt_counter} corrupt files")
    if corrupt_counter == 0:
        return True
    else:
        return False

In [55]:
print("Check train set")
assert(check_valid_pc(train))
print("Check val set")
assert(check_valid_pc(val))
print("Check test set")
check_valid_pc(test)

Check train set
Check val set


100%|█████████████████████████████████████| 8928/8928 [00:02<00:00, 4308.38it/s]

There are 0 corrupt files
Check test set


## Check latent representation

In [ ]:
LATENT_ROOT = os.path.join(DATA_ROOT, "latent/pretrained/results_all_zs_chkp1000.h5")

In [ ]:
file_pattern = "**/*.ply"
all_files = glob(f"{DATA_ROOT}/{file_pattern}", recursive=True)
print(f"Total number of files: {len(all_files)}")
print(f"Missing files: {len(train)+len(val)+len(test)-len(all_files)}")

Corrupt files in (test: 28, val: 36, train: 516) -> Total = 290 anscheinend mehr?? Ja anscheinend gab es 290 Fälle in denen nicht konvertiert werden konnte, aber noch ein paar mehr Fälle wo zwar konvertiert wurde, aber die point cloud an sich halt corrupt ist.

NEXT TIME: Herausfinden, welche PC corrupt ist und welche nicht konvertiert werden konnten (290), damit sollten (516-290) = 226 corrupt sein. Dann eine fertige liste mit funktionierenden point clouds erstellen. Am ende zum beispiel ein script "prepare pointnet++ training data.py" oder sowas erstellen.

In [107]:
print(no_points)

['../data/pc_cad/0056/00566337.ply']


Test: 14 pc's waren entweder point cloud conversion fail oder create cad fail, alle anderen korrekt
Val: 18 pc's waren entweder point cloud conversion fail oder create cad fail, alle anderen korrekt
Train: Irgendwo findet ein Parallels problem statt (0, 78950, 1, 82290)Es ist die file 0011/00116212

Error kommt von process_one -> create_CAD (in visualize.py)

Laut stackoverflow soll man mit ulimit -s stack anschauen und mit ulimit -s \<neuerWert> erhöhen

Chat GPT: "On macOS (and other Unix-like operating systems), the ulimit -s command is used to query or set the stack size limit for processes. The stack size determines the amount of memory allocated for a program's stack, which is used for function calls, local variables, and control flow."



In [113]:
problematic_file = "/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/data/cad_json/0011/00116212.json"
try:
    with open(problematic_file, "r") as f:
        data = json.load(f)
    print("File loaded successfully.")
    print(json.dumps(data, indent=4))  # Pretty-print the JSON content.
except json.JSONDecodeError as e:
    print(f"JSON decoding error: {e}")
except Exception as e:
    print(f"Error reading file: {e}")

File loaded successfully.
{
    "entities": {
        "FRAcwEqNExbvyOz_2": {
            "name": "Extrude 3",
            "type": "ExtrudeFeature",
            "profiles": [
                {
                    "profile": "JNC",
                    "sketch": "FRSAyqbsz52iKa9_2"
                }
            ],
            "extent_two": {
                "distance": {
                    "type": "ModelParameter",
                    "role": "AgainstDistance",
                    "name": "none",
                    "value": 0.0
                },
                "type": "DistanceExtentDefinition",
                "taper_angle": {
                    "type": "ModelParameter",
                    "role": "Side2TaperAngle",
                    "name": "none",
                    "value": 0.0
                }
            },
            "extent_one": {
                "distance": {
                    "type": "ModelParameter",
                    "role": "AlongDistance",
                   

In [36]:
a = []


In [37]:
not a

True